# Comparación reproducible de superresolución sobre SMB

Notebook principal del experimento acelerado: muestra fija de 64 páginas independientes, seis degradaciones controladas, bicúbica, EDSR-baseline y SwinIR-lightweight. SMB se usa exclusivamente para evaluación; no se ajusta ni se elige ningún método con sus resultados.

## Contexto y método

La comparación contrasta una interpolación transparente, una CNN residual y un Transformer ligero. Los checkpoints x2/x4 proceden de los repositorios oficiales y se verifican por SHA-256 antes de cargar. La muestra fue seleccionada por rango SHA-256 antes de observar resultados y contiene 64 grupos fuente distintos.

### Supuestos clave

- EDSR-baseline y SwinIR-lightweight fueron entrenados para degradación bicúbica; las condiciones moderate/strong miden transferencia fuera de esa degradación de entrenamiento.
- Las conclusiones se limitan a la muestra fija; no se estimará prevalencia para las 685 páginas.
- El notebook no abre SMB hasta que existe un desbloqueo científico completo y revisado.

## Preparación del entorno

Local: ejecutar previamente `uv sync --extra cpu --group dev --group notebooks --group kaggle`.

Kaggle: activar GPU e Internet y guardar `HF_TOKEN` en Secrets. La celda clona el mismo repositorio si hace falta e instala el proyecto con `uv`; conserva la versión de PyTorch/CUDA suministrada por Kaggle.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/dreinon/proyecto-tfg.git"
IS_KAGGLE = Path("/kaggle/working").is_dir()


def discover_project_root() -> Path:
    starts = [Path.cwd(), Path("/kaggle/working/proyecto-tfg")]
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / "pyproject.toml").is_file():
                return candidate.resolve()
    if not IS_KAGGLE:
        raise RuntimeError("No se encuentra el repositorio del proyecto.")
    if shutil.which("uv") is None:
        raise RuntimeError("El runtime de Kaggle no incluye uv; instálalo y vuelve a ejecutar.")
    destination = Path("/kaggle/working/proyecto-tfg")
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(destination)], check=True)
    return destination.resolve()


PROJECT_ROOT = discover_project_root()
if IS_KAGGLE:
    if shutil.which("uv") is None:
        raise RuntimeError("uv es obligatorio para preparar el runtime de Kaggle.")
    torch_before = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.__version__)"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    subprocess.run(["uv", "pip", "install", "--system", "-e", str(PROJECT_ROOT)], check=True)
    torch_after = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.__version__)"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if torch_after != torch_before:
        raise RuntimeError("La preparación reemplazó PyTorch; reinicia y revisa el runtime.")

os.chdir(PROJECT_ROOT)
PROJECT_ROOT

In [ ]:
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

from score_super_resolution.benchmark_policy import BenchmarkPurpose, BenchmarkState
from score_super_resolution.comparison import (
    CONDITIONS,
    QUALITATIVE_ASSIGNMENT,
    as_rgb8,
    ensure_manifest_generation,
    fidelity_metrics,
    load_evaluation_sample,
    load_frozen_degradation,
    technical_smoke_image,
)
from score_super_resolution.degradation import align_reference, apply_degradation
from score_super_resolution.pretrained import MODEL_METHODS, PretrainedSRRunner
from score_super_resolution.smb import check_access, load_smb

SEED = 20260829
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

EXPERIMENT_PATH = PROJECT_ROOT / "configs/experiments/smb-pretrained-evaluation-v1.yaml"
UNLOCK_PATH = PROJECT_ROOT / "configs/smb-evaluation-v1/unlock.json"
GENERATION_ROOT = ensure_manifest_generation(PROJECT_ROOT)
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts/phase3-smb-evaluation"
experiment = yaml.safe_load(EXPERIMENT_PATH.read_text(encoding="utf-8"))
evaluation_unlocked = experiment["status"] == "frozen" and UNLOCK_PATH.is_file()
run_smb_evaluation = IS_KAGGLE or os.environ.get("RUN_SMB_EVALUATION") == "1"
evaluation_ready = evaluation_unlocked and run_smb_evaluation
{
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "seed": SEED,
    "evaluation_unlocked": evaluation_unlocked,
    "run_smb_evaluation": run_smb_evaluation,
    "evaluation_ready": evaluation_ready,
}

## Datos

In [ ]:
access = check_access()
sample = load_evaluation_sample(PROJECT_ROOT)
sample_frame = pd.DataFrame([row.__dict__ for row in sample])
pd.DataFrame(
    [
        {
            "repository": access.get("repository_id"),
            "revision": access.get("resolved_revision"),
            "access_status": access.get("status"),
            "descriptor_match": access.get("descriptor_match"),
            "sample_pages": len(sample_frame),
            "independent_source_groups": sample_frame.source_group_id.nunique(),
        }
    ]
)

## Métodos y smoke test técnico

In [ ]:
runner = PretrainedSRRunner(
    PROJECT_ROOT,
    tile_size=int(experiment["inference"]["tile_size"]),
    tile_overlap=int(experiment["inference"]["tile_overlap"]),
)
checkpoint_paths = runner.prepare()
pd.DataFrame(
    [
        {"checkpoint": key, "available": Path(value).is_file()}
        for key, value in checkpoint_paths.items()
    ]
)

In [ ]:
smoke_hr = technical_smoke_image()
smoke_rows = []
for scale in (2, 4):
    condition_id = f"x{scale}-clean"
    smoke_lr = cv2.resize(
        smoke_hr,
        (smoke_hr.shape[1] // scale, smoke_hr.shape[0] // scale),
        interpolation=cv2.INTER_AREA,
    )
    for method_id in MODEL_METHODS:
        result = runner.run(
            method_id, smoke_lr, target_shape=smoke_hr.shape, condition_id=condition_id
        )
        smoke_rows.append(
            {
                "scale": scale,
                "method": method_id,
                "shape_ok": result.pixels.shape == smoke_hr.shape,
                "runtime_seconds": result.elapsed_ns / 1e9,
                "parameters": result.evidence.get("parameter_count"),
            }
        )
smoke_results = pd.DataFrame(smoke_rows)
smoke_results

El smoke test anterior solo comprueba descarga, carga, dimensiones e inferencia; no es evidencia musical ni se utilizará en las conclusiones.

## Ejecución SMB

In [ ]:
if not evaluation_ready:
    reason = (
        "falta el desbloqueo científico"
        if not evaluation_unlocked
        else "ejecución local desactivada; usa Kaggle o RUN_SMB_EVALUATION=1"
    )
    print(f"Evaluación SMB no iniciada: {reason}.")
    dataset = None
else:
    unlock = json.loads(UNLOCK_PATH.read_text(encoding="utf-8"))
    dataset = load_smb(
        purpose=BenchmarkPurpose.INFERENCE,
        state=BenchmarkState.EVALUATION_UNLOCKED,
        unlock_record=unlock,
        project_root=PROJECT_ROOT,
        manifest_generation_root=GENERATION_ROOT,
    )
    print(f"SMB cargado en la revisión fijada: {len(dataset)} páginas.")

In [ ]:
RESULTS_PATH = ARTIFACT_ROOT / "raw-metrics.csv"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
results = pd.read_csv(RESULTS_PATH) if RESULTS_PATH.is_file() else pd.DataFrame()

if dataset is not None:
    degradation = load_frozen_degradation(PROJECT_ROOT)
    completed = (
        set()
        if results.empty
        else set(zip(results.item_id, results.condition_id, results.method_id, strict=True))
    )
    records = results.to_dict("records") if not results.empty else []
    qualitative_lookup = dict(QUALITATIVE_ASSIGNMENT)
    for sample_row in sample:
        hr_original = as_rgb8(dataset[sample_row.upstream_index]["image"])
        for condition_id in CONDITIONS:
            scale = int(condition_id[1])
            degraded = apply_degradation(
                hr_original,
                control=degradation,
                condition_id=condition_id,
                item_id=sample_row.item_id,
                source_group_id=sample_row.source_group_id,
                fixture_manifest_id="smb-pretrained-evaluation-v1",
                purpose="benchmark",
            )
            reference = align_reference(hr_original, scale).pixels
            for method_id in MODEL_METHODS:
                key = (sample_row.item_id, condition_id, method_id)
                if key in completed:
                    continue
                reconstruction = runner.run(
                    method_id,
                    degraded.pixels,
                    target_shape=reference.shape,
                    condition_id=condition_id,
                )
                metrics = fidelity_metrics(reference, reconstruction.pixels)
                records.append(
                    {
                        "upstream_index": sample_row.upstream_index,
                        "item_id": sample_row.item_id,
                        "source_group_id": sample_row.source_group_id,
                        "condition_id": condition_id,
                        "scale": scale,
                        "profile": condition_id.split("-", 1)[1],
                        "method_id": method_id,
                        **metrics,
                        "runtime_seconds": reconstruction.elapsed_ns / 1e9,
                        "degradation_trace_id": degraded.trace["trace_id"],
                        "output_sha256": reconstruction.evidence["output_pixel_sha256"],
                        "checkpoint_sha256": reconstruction.evidence.get("checkpoint_sha256"),
                    }
                )
                if qualitative_lookup.get(sample_row.item_id) == condition_id:
                    output_dir = ARTIFACT_ROOT / "qualitative" / sample_row.item_id / condition_id
                    output_dir.mkdir(parents=True, exist_ok=True)
                    cv2.imwrite(
                        str(output_dir / f"{method_id}.png"),
                        cv2.cvtColor(reconstruction.pixels, cv2.COLOR_RGB2BGR),
                    )
            results = pd.DataFrame(records)
        temporary = RESULTS_PATH.with_suffix(".csv.tmp")
        results.to_csv(temporary, index=False)
        os.replace(temporary, RESULTS_PATH)
    print(
        f"Tuplas reconciliadas: {len(results)} / "
        f"{len(sample) * len(CONDITIONS) * len(MODEL_METHODS)}"
    )
results.head() if not results.empty else results

## Resultados

In [ ]:
if results.empty:
    aggregate = pd.DataFrame()
else:
    aggregate = results.groupby(["condition_id", "method_id"], as_index=False).agg(
        pages=("item_id", "nunique"),
        psnr_y_mean=("psnr_y", "mean"),
        ssim_y_mean=("ssim_y", "mean"),
        psnr_rgb_mean=("psnr_rgb", "mean"),
        ssim_rgb_mean=("ssim_rgb", "mean"),
        runtime_seconds_median=("runtime_seconds", "median"),
    )
aggregate

In [ ]:
if not aggregate.empty:
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for method_id, frame in aggregate.groupby("method_id"):
        axes[0].plot(frame.condition_id, frame.psnr_y_mean, marker="o", label=method_id)
        axes[1].plot(frame.condition_id, frame.ssim_y_mean, marker="o", label=method_id)
    axes[0].set(title="PSNR medio en luminancia", ylabel="dB", xlabel="Condición")
    axes[1].set(title="SSIM medio en luminancia", ylabel="SSIM", xlabel="Condición")
    for axis in axes:
        axis.tick_params(axis="x", rotation=35)
        axis.grid(alpha=0.25)
    axes[1].legend(fontsize=8)
    figure.tight_layout()
else:
    print("Todavía no hay resultados SMB que representar.")

## Conclusiones

Las conclusiones se redactarán únicamente después de ejecutar la matriz completa, reconciliar las 1.152 tuplas y revisar las seis comparaciones cualitativas preasignadas. Hasta entonces no se promoverá ninguna afirmación sobre calidad, fidelidad musical o uso profesional.